# Stage 1 - 05: V-JEPA 2.1-B Multi-Clip Evaluation

A7 `V-JEPA 2.1-B` achieved a strong DACON Stage 1 public score (`0.9549731183`) with a **single 16-frame centre clip** and threshold `0.5`.

This notebook does **no training**. It loads the existing A7 `best.pt` and evaluates the same model with `1 / 3 / 5` deterministic validation clips per video using the repository's own `build_clip_sampler` + `Stage1Evaluator` pipeline.

Goals:
- reproduce the original 1-clip A7 validation result first (sanity gate),
- measure whether 3/5 temporal clips improve Macro-F1 at the fixed `0.5` threshold,
- record tuned-threshold Macro-F1 only as a diagnostic,
- measure runtime cost,
- save separate prediction CSVs without overwriting A7 outputs.

The default model-selection rule is conservative: **highest Macro-F1 @ 0.5, then fewer clips on ties**.


## 1. Setup


In [1]:
from __future__ import annotations

import json
import os
import subprocess
import sys
import time
from pathlib import Path

REPO_URL = "https://github.com/sangchun1/Blackbox-Detection.git"
BRANCH = "stage1-sangchun"

IN_COLAB = False
try:
    from google.colab import drive

    IN_COLAB = True
    drive.mount("/content/drive", force_remount=False)
except ModuleNotFoundError:
    print("Not running in Google Colab; Drive mount skipped.")

if IN_COLAB:
    REPO_ROOT = Path("/content/Blackbox-Detection")

    if not (REPO_ROOT / ".git").is_dir():
        subprocess.run(
            [
                "git", "clone", "--depth", "1", "--branch", BRANCH,
                "--single-branch", REPO_URL, str(REPO_ROOT),
            ],
            check=True,
        )
    else:
        current_branch = subprocess.run(
            ["git", "-C", str(REPO_ROOT), "branch", "--show-current"],
            check=True,
            capture_output=True,
            text=True,
        ).stdout.strip()

        if current_branch != BRANCH:
            subprocess.run(
                ["git", "-C", str(REPO_ROOT), "checkout", BRANCH],
                check=True,
            )

        dirty = subprocess.run(
            ["git", "-C", str(REPO_ROOT), "status", "--porcelain"],
            check=True,
            capture_output=True,
            text=True,
        ).stdout.strip()

        if dirty:
            print("WARNING: local repo has changes; git pull skipped.")
        else:
            subprocess.run(
                ["git", "-C", str(REPO_ROOT), "pull", "--ff-only", "origin", BRANCH],
                check=True,
            )
else:
    REPO_ROOT = Path.cwd().resolve()
    while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "pyproject.toml").is_file():
        REPO_ROOT = REPO_ROOT.parent
    if not (REPO_ROOT / "pyproject.toml").is_file():
        raise FileNotFoundError("Run this notebook inside Blackbox-Detection repository.")

os.chdir(REPO_ROOT)

# Keep Colab's binary scientific stack; install Stage 1 extras only.
COLAB_EXTRAS = [
    "av>=15,<17",
    "timm==1.0.15",
    "fvcore==0.1.5.post20221221",
    "iopath==0.1.10",
    "yacs==0.1.8",
    "einops==0.8.1",
    "omegaconf==2.3.0",
    "hydra-core==1.3.2",
    "easydict==1.13",
]

if IN_COLAB:
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "--upgrade-strategy", "only-if-needed", *COLAB_EXTRAS,
        ],
        check=True,
    )

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(REPO_ROOT)],
    check=True,
)

if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

import numpy as np
import pandas as pd
import torch
import yaml

from blackbox_detection.utils import load_checkpoint, seed_everything
from blackbox_detection.stage1.dataset import Stage1VideoDataset, build_dataloader, video_batch_adapter
from blackbox_detection.stage1.evaluator import AggregationConfig, Stage1Evaluator, save_predictions
from blackbox_detection.stage1.models import build_stage1_model, count_parameters
from blackbox_detection.stage1.sampling import build_clip_sampler
from blackbox_detection.stage1.transforms import ClipAugmentConfig, build_video_transforms

DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/Blackbox-Detection")
DATASET_ROOT = DRIVE_PROJECT_ROOT / "DATASET"
DLC_ROOT = DATASET_ROOT / "DLC-2021"
DLC_SPLIT_CSV = DLC_ROOT / "dlc_split.csv"
OUTPUT_ROOT = DRIVE_PROJECT_ROOT / "outputs" / "stage1"
CONFIG_DIR = REPO_ROOT / "configs" / "stage1"

GIT_COMMIT = subprocess.run(
    ["git", "-C", str(REPO_ROOT), "rev-parse", "--short", "HEAD"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()

print("repo   :", REPO_ROOT)
print("commit :", GIT_COMMIT)
print("torch  :", torch.__version__)
print("cuda   :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu    :", torch.cuda.get_device_name(0))


Mounted at /content/drive
repo   : /content/Blackbox-Detection
commit : 3597d69
torch  : 2.8.0+cu126
cuda   : True
gpu    : NVIDIA L4


## 2. Experiment settings


In [2]:
# Existing trained A7 run. This notebook never overwrites it.
RESULT_VARIANT = "dlc"
MODEL_NAME = "vjepa2_1_b"
BASE_RUN_DIR = OUTPUT_ROOT / RESULT_VARIANT / MODEL_NAME
BEST_CKPT = BASE_RUN_DIR / "best.pt"
BASE_SUMMARY = BASE_RUN_DIR / "summary.json"

# New 05-only outputs.
MULTICLIP_DIR = OUTPUT_ROOT / RESULT_VARIANT / "vjepa2_1_b_multiclip"
MULTICLIP_DIR.mkdir(parents=True, exist_ok=True)

# Main ablation. Change to [1, 3] if you want a quicker first pass.
CLIP_COUNTS = [1, 3, 5]

# Keep exactly the training config's validation stride unless explicitly overridden.
VAL_STRIDE_OVERRIDE = None

if not BEST_CKPT.is_file():
    raise FileNotFoundError(f"A7 best checkpoint not found: {BEST_CKPT}")

print("A7 checkpoint :", BEST_CKPT)
print("05 output     :", MULTICLIP_DIR)
print("clip counts   :", CLIP_COUNTS)


A7 checkpoint : /content/drive/MyDrive/Blackbox-Detection/outputs/stage1/dlc/vjepa2_1_b/best.pt
05 output     : /content/drive/MyDrive/Blackbox-Detection/outputs/stage1/dlc/vjepa2_1_b_multiclip
clip counts   : [1, 3, 5]


## 3. V-JEPA 2.1 source + checkpoint + model config


In [3]:
CONFIG = yaml.safe_load((CONFIG_DIR / "vjepa2_1_b.yaml").read_text(encoding="utf-8"))
assert CONFIG["model"]["name"] == MODEL_NAME

SEED = int(CONFIG["train"]["seed"])
seed_everything(SEED, deterministic=False)
ADAPTER = video_batch_adapter()

# The wrapper imports the official facebookresearch/vjepa2 source tree.
VJEPA_SOURCE_ROOT = Path("/content/vjepa2")
VJEPA_CKPT_DIR = DRIVE_PROJECT_ROOT / "pretrained" / "vjepa2"
VJEPA_CKPT_DIR.mkdir(parents=True, exist_ok=True)
VJEPA_CKPT = VJEPA_CKPT_DIR / "vjepa2_1_vitb_dist_vitG_384.pt"

if not (VJEPA_SOURCE_ROOT / ".git").is_dir():
    print("Cloning official V-JEPA 2 source...")
    subprocess.run(
        [
            "git", "clone", "--depth", "1",
            "https://github.com/facebookresearch/vjepa2.git",
            str(VJEPA_SOURCE_ROOT),
        ],
        check=True,
    )

if not VJEPA_CKPT.is_file():
    print("Downloading official V-JEPA 2.1-B checkpoint to Drive...")
    subprocess.run(
        [
            "wget", "-O", str(VJEPA_CKPT),
            "https://dl.fbaipublicfiles.com/vjepa2/vjepa2_1_vitb_dist_vitG_384.pt",
        ],
        check=True,
    )

params = CONFIG["model"]["params"]
params["source_root"] = str(VJEPA_SOURCE_ROOT)
params["checkpoint_path"] = str(VJEPA_CKPT)
params["allow_download"] = False

print("source root :", params["source_root"])
print("pretrained  :", params["checkpoint_path"])
print("best.pt     :", BEST_CKPT)


Cloning official V-JEPA 2 source...
source root : /content/vjepa2
pretrained  : /content/drive/MyDrive/Blackbox-Detection/pretrained/vjepa2/vjepa2_1_vitb_dist_vitG_384.pt
best.pt     : /content/drive/MyDrive/Blackbox-Detection/outputs/stage1/dlc/vjepa2_1_b/best.pt


## 4. DLC validation manifest


In [4]:
VIDEO_EXTENSIONS = {
    ".mp4", ".mov", ".avi", ".mkv", ".m4v", ".webm",
}


def _normalize_rel_text(value: str) -> str:
    return str(value).replace("\\", "/").strip().lstrip("./")


def _build_dlc_video_index() -> dict[tuple[str, str], str]:
    index: dict[tuple[str, str], str] = {}
    source_counts: dict[str, int] = {}

    for source in ("or", "re"):
        clips_root = DLC_ROOT / source / "clips_video"
        if not clips_root.is_dir():
            raise FileNotFoundError(f"DLC clips directory not found: {clips_root}")

        count = 0
        for path in clips_root.rglob("*"):
            if not path.is_file() or path.suffix.lower() not in VIDEO_EXTENSIONS:
                continue

            rel_no_suffix = path.relative_to(clips_root).with_suffix("").as_posix()
            key = (source, _normalize_rel_text(rel_no_suffix))

            if key in index and index[key] != str(path):
                raise ValueError(f"Duplicate DLC video key: {key}")

            index[key] = str(path)
            count += 1

        source_counts[source] = count

    print("indexed DLC videos:", source_counts)
    return index


DLC_VIDEO_INDEX = _build_dlc_video_index()


def _resolve_dlc_video(source: str, clip_id: str) -> str:
    source = str(source).strip().lower()
    clip_id = _normalize_rel_text(clip_id)
    exact_key = (source, clip_id)

    if exact_key in DLC_VIDEO_INDEX:
        return DLC_VIDEO_INDEX[exact_key]

    matches = []
    for (indexed_source, rel_key), video_path in DLC_VIDEO_INDEX.items():
        if indexed_source != source:
            continue
        if rel_key.startswith(clip_id + "/") or rel_key.endswith("/" + clip_id) or rel_key == clip_id:
            matches.append(video_path)

    if len(matches) == 1:
        return matches[0]

    leaf = Path(clip_id).name
    leaf_matches = [
        video_path
        for (indexed_source, rel_key), video_path in DLC_VIDEO_INDEX.items()
        if indexed_source == source and Path(rel_key).name == leaf
    ]
    if len(leaf_matches) == 1:
        return leaf_matches[0]

    raise FileNotFoundError(
        f"Could not uniquely resolve DLC clip_id={clip_id!r}, source={source!r}"
    )


def load_dlc_manifest(split_name: str) -> pd.DataFrame:
    raw = pd.read_csv(DLC_SPLIT_CSV).copy()

    required = {
        "clip_id", "class", "source", "document_type", "document_id",
        "group", "split", "device", "condition",
    }
    missing = sorted(required - set(raw.columns))
    if missing:
        raise ValueError(f"dlc_split.csv missing columns: {missing}")

    raw["split"] = raw["split"].astype(str).str.strip().str.lower()
    raw["source"] = raw["source"].astype(str).str.strip().str.lower()
    raw["class"] = raw["class"].astype(str).str.strip().str.lower()

    label_map = {"original": "ORIGINAL", "rerecorded": "RERECORDED"}
    frame = raw.loc[raw["split"].eq(split_name)].copy()
    if frame.empty:
        raise ValueError(f"No DLC rows for split={split_name!r}")

    frame["label"] = frame["class"].map(label_map)
    if frame["label"].isna().any():
        raise ValueError("Unexpected DLC class value found.")

    frame["video_id"] = "dlc__" + frame["clip_id"].astype(str).str.replace("/", "__", regex=False)
    frame["dataset"] = "dlc2021"
    frame["scene_type"] = "document"
    frame["video_path"] = [
        _resolve_dlc_video(source, clip_id)
        for source, clip_id in zip(frame["source"], frame["clip_id"])
    ]

    keep = [
        "video_path", "label", "video_id", "dataset", "scene_type",
        "clip_id", "source", "document_type", "document_id", "group",
        "device", "condition",
    ]
    return frame[keep].reset_index(drop=True)


val_df = load_dlc_manifest("val")
missing_val = [p for p in val_df["video_path"] if not Path(p).is_file()]
if missing_val:
    raise FileNotFoundError(f"Missing validation videos: {len(missing_val)}")

print("validation videos:", len(val_df))
display(val_df["label"].value_counts().rename("count").to_frame())


indexed DLC videos: {'or': 290, 're': 400}
validation videos: 104


,count
label,
RERECORDED,60
ORIGINAL,44


## 5. Build A7 model and restore `best.pt`


In [5]:
model = build_stage1_model(
    MODEL_NAME,
    finetune_mode=CONFIG["model"]["finetune_mode"],
    unfreeze_last_n=int(CONFIG["model"]["unfreeze_last_n"]),
    **params,
)

# Load from CPU to avoid checkpoint RNG/device issues; RNG restoration is unnecessary for deterministic eval.
load_checkpoint(
    BEST_CKPT,
    model=model,
    map_location="cpu",
    restore_rng_state=False,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device).eval()

print("device      :", device)
print("parameters  :", count_parameters(model))
print("preprocess  :", model.preprocessing())


/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


device      : cuda
parameters  : {'total': 89199362, 'trainable': 2366210, 'frozen': 86833152}
preprocess  : {'input_kind': 'video', 'num_frames': 16, 'input_size': 384, 'mean': (0.485, 0.456, 0.406), 'std': (0.229, 0.224, 0.225), 'channels_first': True, 'source': 'https://dl.fbaipublicfiles.com/vjepa2/vjepa2_1_vitb_dist_vitG_384.pt', 'notes': 'Official V-JEPA 2.1 preprocessing: shorter side resized to crop_size * 256 / 224, centre crop, ImageNet normalisation. The released 2.1 encoders are 384-resolution.'}


## 6. Build validation transform


In [6]:
video_config = CONFIG["data"]
augmentation_config = CONFIG["augmentation"]
preprocessing = model.preprocessing()

_, val_transform = build_video_transforms(
    crop_size=int(preprocessing["input_size"]),
    mean=tuple(preprocessing["mean"]),
    std=tuple(preprocessing["std"]),
    train_config=ClipAugmentConfig(
        crop_size=int(preprocessing["input_size"]),
        scale_range=tuple(augmentation_config["scale_range"]),
        ratio_range=tuple(augmentation_config["ratio_range"]),
        hflip_prob=float(augmentation_config["hflip_prob"]),
        brightness=float(augmentation_config["brightness"]),
        contrast=float(augmentation_config["contrast"]),
        perspective_prob=float(augmentation_config["perspective_prob"]),
        perspective_scale=float(augmentation_config["perspective_scale"]),
    ),
)

VAL_STRIDE = (
    int(video_config["val_stride"])
    if VAL_STRIDE_OVERRIDE is None
    else int(VAL_STRIDE_OVERRIDE)
)

AGGREGATION = AggregationConfig(
    frame_method=CONFIG["evaluation"]["aggregation"]["frame_method"],
    video_method=CONFIG["evaluation"]["aggregation"]["video_method"],
)

AMP = bool(CONFIG["train"]["amp"])

print("num frames :", int(video_config["num_frames"]))
print("val stride :", VAL_STRIDE)
print("AMP        :", AMP)
print("aggregation:", AGGREGATION)


num frames : 16
val stride : 2
AMP        : True
aggregation: AggregationConfig(frame_method='mean', video_method='mean')


## 7. Multi-clip evaluation: 1 / 3 / 5 clips


In [7]:
def evaluate_num_clips(num_clips: int):
    dataset = Stage1VideoDataset(
        val_df,
        clip_sampler=build_clip_sampler(
            train=False,
            num_frames=int(video_config["num_frames"]),
            val_stride=VAL_STRIDE,
            num_clips=int(num_clips),
        ),
        transform=val_transform,
        on_error="zero",
        deterministic=True,
    )

    loader = build_dataloader(
        dataset,
        batch_size=int(video_config["val_batch_size"]),
        shuffle=False,
        num_workers=int(video_config["num_workers"]),
        seed=SEED,
    )

    evaluator = Stage1Evaluator(
        model,
        ADAPTER,
        device=device,
        amp=AMP,
        aggregation=AGGREGATION,
    )

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()

    started = time.perf_counter()
    result, units = evaluator.evaluate(loader, return_units=True)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    elapsed = time.perf_counter() - started
    peak_gb = (
        torch.cuda.max_memory_allocated() / 1024**3
        if torch.cuda.is_available()
        else float("nan")
    )

    pred_path = MULTICLIP_DIR / f"val_predictions_{num_clips}clip.csv"
    save_predictions(result.predictions, pred_path)

    row = {
        "num_clips": int(num_clips),
        "macro_f1": float(result.macro_f1),
        "macro_f1_at_0.5": float(result.macro_f1_at_default),
        "best_threshold": float(result.threshold),
        "f1_original": float(result.per_class_f1.get("ORIGINAL", float("nan"))),
        "f1_rerecorded": float(result.per_class_f1.get("RERECORDED", float("nan"))),
        "num_videos": int(result.num_videos),
        "num_invalid_videos": int(result.num_invalid_videos),
        "elapsed_seconds": float(elapsed),
        "seconds_per_video": float(elapsed / max(result.num_videos, 1)),
        "peak_gpu_gb": float(peak_gb),
        "prediction_file": str(pred_path),
    }
    return row, result, units


rows = []
results_by_clips = {}
units_by_clips = {}

for n in CLIP_COUNTS:
    print("\n" + "=" * 70)
    print(f"Evaluating {n} clip(s) per video")
    print("=" * 70)

    row, result, units = evaluate_num_clips(n)
    rows.append(row)
    results_by_clips[n] = result
    units_by_clips[n] = units

    print(f"Macro-F1 tuned : {row['macro_f1']:.6f}")
    print(f"Macro-F1 @0.5  : {row['macro_f1_at_0.5']:.6f}")
    print(f"best threshold : {row['best_threshold']:.6f}")
    print(f"runtime        : {row['elapsed_seconds'] / 60:.2f} min")
    print(f"peak GPU       : {row['peak_gpu_gb']:.2f} GB")

summary_df = pd.DataFrame(rows).sort_values("num_clips").reset_index(drop=True)
summary_df.to_csv(MULTICLIP_DIR / "multiclip_summary.csv", index=False)
display(summary_df)



Evaluating 1 clip(s) per video


/usr/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)


Macro-F1 tuned : 1.000000
Macro-F1 @0.5  : 1.000000
best threshold : 0.500000
runtime        : 2.39 min
peak GPU       : 0.52 GB

Evaluating 3 clip(s) per video


/usr/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)


Macro-F1 tuned : 1.000000
Macro-F1 @0.5  : 1.000000
best threshold : 0.500000
runtime        : 4.47 min
peak GPU       : 0.88 GB

Evaluating 5 clip(s) per video


/usr/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)


Macro-F1 tuned : 1.000000
Macro-F1 @0.5  : 1.000000
best threshold : 0.500000
runtime        : 7.23 min
peak GPU       : 1.24 GB


,num_clips,macro_f1,macro_f1_at_0.5,best_threshold,f1_original,f1_rerecorded,num_videos,num_invalid_videos,elapsed_seconds,seconds_per_video,peak_gpu_gb,prediction_file
0,1,1.0,1.0,0.5,1.0,1.0,104,0,143.343304,1.378301,0.523580,/content/drive/MyDrive/Blackbox-Detection/outp...
1,3,1.0,1.0,0.5,1.0,1.0,104,0,268.368708,2.580468,0.878072,/content/drive/MyDrive/Blackbox-Detection/outp...
2,5,1.0,1.0,0.5,1.0,1.0,104,0,433.578150,4.169021,1.236470,/content/drive/MyDrive/Blackbox-Detection/outp...


## 8. Sanity gate: 1-clip must reproduce A7


In [8]:
if 1 not in results_by_clips:
    raise RuntimeError("CLIP_COUNTS must include 1 for the A7 sanity check.")

one = summary_df.loc[summary_df["num_clips"].eq(1)].iloc[0]

if BASE_SUMMARY.is_file():
    baseline = json.loads(BASE_SUMMARY.read_text(encoding="utf-8"))
    expected_tuned = float(baseline["val_macro_f1"])
    expected_default = float(baseline["val_macro_f1_at_0.5"])

    tuned_diff = abs(float(one["macro_f1"]) - expected_tuned)
    default_diff = abs(float(one["macro_f1_at_0.5"]) - expected_default)

    print("saved A7 tuned    :", expected_tuned)
    print("05 1-clip tuned   :", float(one["macro_f1"]))
    print("saved A7 @0.5     :", expected_default)
    print("05 1-clip @0.5    :", float(one["macro_f1_at_0.5"]))

    if tuned_diff > 1e-8 or default_diff > 1e-8:
        raise RuntimeError(
            "1-clip result does not reproduce the saved A7 validation result. "
            "Do not interpret the multi-clip comparison until config/data/code mismatch is fixed."
        )

    print("[PASS] 1-clip exactly reproduces saved A7 validation metrics.")
else:
    print("WARNING: A7 summary.json not found; automatic exact-match check skipped.")


saved A7 tuned    : 1.0
05 1-clip tuned   : 1.0
saved A7 @0.5     : 1.0
05 1-clip @0.5    : 1.0
[PASS] 1-clip exactly reproduces saved A7 validation metrics.


## 9. Conservative selection and runtime view


In [9]:
# Primary selection uses the actual default inference boundary (0.5), not a threshold
# re-tuned on the same 104 validation videos. On ties, prefer fewer clips.
ranking = summary_df.sort_values(
    ["macro_f1_at_0.5", "num_clips"],
    ascending=[False, True],
    kind="mergesort",
).reset_index(drop=True)

display(ranking[
    [
        "num_clips",
        "macro_f1_at_0.5",
        "macro_f1",
        "best_threshold",
        "elapsed_seconds",
        "seconds_per_video",
        "peak_gpu_gb",
    ]
])

best = ranking.iloc[0]
recommendation = {
    "selection_rule": "highest Macro-F1 @ 0.5; fewer clips on ties",
    "recommended_num_clips": int(best["num_clips"]),
    "val_macro_f1_at_0.5": float(best["macro_f1_at_0.5"]),
    "diagnostic_tuned_macro_f1": float(best["macro_f1"]),
    "diagnostic_best_threshold": float(best["best_threshold"]),
    "elapsed_seconds_on_dlc_val": float(best["elapsed_seconds"]),
    "a7_public_anchor": 0.9549731183,
    "note": (
        "Do not submit automatically from this notebook. "
        "Use runtime + fixed-threshold validation evidence to decide whether to build a multi-clip inference.py."
    ),
}

(MULTICLIP_DIR / "multiclip_recommendation.json").write_text(
    json.dumps(recommendation, indent=2),
    encoding="utf-8",
)

print(json.dumps(recommendation, indent=2))
print("\nsaved:")
print(" -", MULTICLIP_DIR / "multiclip_summary.csv")
print(" -", MULTICLIP_DIR / "multiclip_recommendation.json")
for n in CLIP_COUNTS:
    print(" -", MULTICLIP_DIR / f"val_predictions_{n}clip.csv")


,num_clips,macro_f1_at_0.5,macro_f1,best_threshold,elapsed_seconds,seconds_per_video,peak_gpu_gb
0,1,1.0,1.0,0.5,143.343304,1.378301,0.523580
1,3,1.0,1.0,0.5,268.368708,2.580468,0.878072
2,5,1.0,1.0,0.5,433.578150,4.169021,1.236470


{
  "selection_rule": "highest Macro-F1 @ 0.5; fewer clips on ties",
  "recommended_num_clips": 1,
  "val_macro_f1_at_0.5": 1.0,
  "diagnostic_tuned_macro_f1": 1.0,
  "diagnostic_best_threshold": 0.5,
  "elapsed_seconds_on_dlc_val": 143.34330439199994,
  "a7_public_anchor": 0.9549731183,
  "note": "Do not submit automatically from this notebook. Use runtime + fixed-threshold validation evidence to decide whether to build a multi-clip inference.py."
}

saved:
 - /content/drive/MyDrive/Blackbox-Detection/outputs/stage1/dlc/vjepa2_1_b_multiclip/multiclip_summary.csv
 - /content/drive/MyDrive/Blackbox-Detection/outputs/stage1/dlc/vjepa2_1_b_multiclip/multiclip_recommendation.json
 - /content/drive/MyDrive/Blackbox-Detection/outputs/stage1/dlc/vjepa2_1_b_multiclip/val_predictions_1clip.csv
 - /content/drive/MyDrive/Blackbox-Detection/outputs/stage1/dlc/vjepa2_1_b_multiclip/val_predictions_3clip.csv
 - /content/drive/MyDrive/Blackbox-Detection/outputs/stage1/dlc/vjepa2_1_b_multiclip/val_pred

## 10. How to interpret the result

Use `Macro-F1 @ 0.5` as the main signal because A7's public success also used the natural `0.5` decision boundary.

- **3 clips > 1 clip at 0.5**: strong candidate for a new submission inference path. Benchmark total runtime before submitting.
- **3 clips = 1 clip**: prefer 1 clip unless confidence / robustness analysis later shows a clear reason to pay the runtime cost.
- **5 clips only improves tuned-threshold F1**: do not treat that alone as evidence for public improvement; it can be validation overfitting.
- **1 clip sanity gate fails**: stop. The comparison is invalid until model/data/preprocessing mismatch is fixed.

This notebook deliberately keeps the A7 checkpoint unchanged and writes all outputs to `outputs/stage1/dlc/vjepa2_1_b_multiclip/`.
